# Cryogenic — Offline GPU Music Visualizer

Renders a polished 1080p60 music video from your stems-analysis JSON and the original mp3, using Colab's GPU.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is fine; L4 is faster).

Workflow:
1. Run the **Setup** cell once (~1 minute).
2. Run the **Upload** cell and drop in your `viz_data_v2.json` and the matching `.mp3`.
3. Run **Render**. ~10–25 min for a 3-min song at 1080p60 on a T4.
4. Preview / download `cryogenic.mp4` from the last cell.

## 1. Setup — install GPU OpenGL + ffmpeg + clone repo

In [1]:
%%bash
set -e
# Headless EGL + GL libs (Colab GPU runtime already has NVIDIA drivers)
apt-get -qq update
apt-get -qq install -y libegl1 libgles2 libglvnd0 libgl1 libglib2.0-0 ffmpeg >/dev/null
pip -q install moderngl==5.10.0 numpy

# Pull the renderer source.
if [ ! -d /content/music-viz ]; then
  git clone -q https://github.com/splice11/music-viz.git /content/music-viz
fi
cd /content/music-viz
git fetch -q origin
git checkout -q claude/enhance-music-visualization-BB1Ui
git pull -q origin claude/enhance-music-visualization-BB1Ui

echo 'Setup complete.'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.9/270.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 5.3 MB/s eta 0:00:00
Setup complete.


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
# Sanity-check that the GPU EGL context comes up.
import sys
sys.path.insert(0, '/content/music-viz')
import moderngl
ctx = moderngl.create_context(standalone=True, backend='egl', require=330)
print('GL renderer:', ctx.info.get('GL_RENDERER'))
print('GL version :', ctx.info.get('GL_VERSION'))
ctx.release()

GL renderer: llvmpipe (LLVM 15.0.7, 256 bits)
GL version : 4.5 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.3


## 2. Upload your song + analysis JSON

Drop in **both** files: the `.mp3` and the matching `viz_data_v2.json`.

In [3]:
import os, shutil
from google.colab import files
os.makedirs('/content/inputs', exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    shutil.move(name, f'/content/inputs/{name}')
    print('saved →', f'/content/inputs/{name}')

Saving Cryogenic.mp3 to Cryogenic.mp3
Saving viz_data_v2.json to viz_data_v2.json
saved → /content/inputs/Cryogenic.mp3
saved → /content/inputs/viz_data_v2.json


In [4]:
import os, glob
mp3s = sorted(glob.glob('/content/inputs/*.mp3') + glob.glob('/content/inputs/*.wav') + glob.glob('/content/inputs/*.m4a'))
jsons = sorted(glob.glob('/content/inputs/*.json'))
assert mp3s,  'No audio file found in /content/inputs (mp3/wav/m4a).'
assert jsons, 'No JSON file found in /content/inputs.'
AUDIO_PATH = mp3s[0]
JSON_PATH  = jsons[0]
print('Audio:', AUDIO_PATH)
print('JSON :', JSON_PATH)

Audio: /content/inputs/Cryogenic.mp3
JSON : /content/inputs/viz_data_v2.json


## 3. Render — this is the long step

Defaults: 1920×1080 @ 60 fps, CRF 17 (visually lossless-ish). Drop to 1280×720 for a quick preview.

In [ ]:
WIDTH       = 1920
HEIGHT      = 1080
TARGET_FPS  = 60
OUTPUT_PATH = '/content/cryogenic.mp4'

import importlib, sys
for m in list(sys.modules):
    if m.startswith('colab_render'):
        del sys.modules[m]
import colab_render

print('Building feature timeline…')
fd = colab_render.build(JSON_PATH, target_fps=TARGET_FPS)
print(f'  {fd.n_frames} frames @ {fd.fps:.0f} fps  ({fd.duration:.1f} s)')

print('Rendering…')
colab_render.render_to_video(
    fd,
    audio_path=AUDIO_PATH,
    out_path=OUTPUT_PATH,
    width=WIDTH, height=HEIGHT,
    crf=17, preset='medium',
    progress_every=120,
)

Building feature timeline…
  12862 frames @ 60 fps  (214.4 s)
Rendering…
  frame 120/12862  1.0 fps  elapsed 122.5s  eta 13011.5s
  frame 240/12862  1.0 fps  elapsed 250.4s  eta 13168.7s
  frame 360/12862  1.0 fps  elapsed 378.6s  eta 13148.5s
  frame 480/12862  0.9 fps  elapsed 505.7s  eta 13046.1s
  frame 600/12862  0.9 fps  elapsed 634.0s  eta 12957.6s
  frame 720/12862  0.9 fps  elapsed 760.8s  eta 12830.8s


## 4. Preview + download

In [ ]:
from IPython.display import HTML
from base64 import b64encode
data = open(OUTPUT_PATH, 'rb').read()
src = 'data:video/mp4;base64,' + b64encode(data).decode()
HTML(f'<video width=720 controls src="{src}"></video>')

In [ ]:
from google.colab import files
files.download(OUTPUT_PATH)